# Inspection des données brutes

**Objectif :** examiner la qualité des données, vérifier les relations entre les cinq tables et définir les règles de nettoyage.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src" / "config.py").is_file():
    raise RuntimeError("Ouvrir ce notebook depuis la racine du projet ou notebooks/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import pandas as pd
from src.cleaning import load_tables, merge_tables, exclude, target
from src.config import CLEAN_INSCRIPTIONS, CLEAN_EVALUATIONS, CLEAN_ACTIVITES, DATA_SUMMARY

In [ ]:
raw = load_tables()
students = raw["etudiants"]
programs = raw["programmes"]
inscriptions = raw["inscriptions"]
evaluations = raw["evaluations"]
activities = raw["activites_virtuelles"]

In [ ]:
tables = raw

### Vérification des clés primaires

In [ ]:
ids = {
    "students": (students, "id_etudiant"),
    "programs": (programs, "id_programme"),
    "inscriptions": (inscriptions, "id_inscription"),
    "evaluations": (evaluations, "id_evaluation"),
    "activities": (activities, "id_activite"),
}
summary = []
for name, (table, id_col) in ids.items():
    summary.append({
        "Table": name,
        "id Column": id_col,
        "Total": len(table),
        "Null": table[id_col].isna().sum(),
        "Duplicate": table[id_col].duplicated().sum(),
        "Unique": table[id_col].is_unique,
    })
summary_df = pd.DataFrame(summary)
summary_df

In [ ]:
print(inscriptions.duplicated(subset=['id_etudiant', 'id_programme']).sum())

### Vérification des clés étrangères

In [ ]:
foreign_keys = {
    "Inscriptions → étudiants": (inscriptions.id_etudiant, students.id_etudiant),
    "Inscriptions → programmes": (inscriptions.id_programme, programs.id_programme),
    "Évaluations → inscriptions": (evaluations.id_inscription, inscriptions.id_inscription),
    "Activités → inscriptions": (activities.id_inscription, inscriptions.id_inscription),
}
pd.DataFrame([
    {"Relation": name, "Clés étrangères invalides": int((~keys.isin(parent)).sum())}
    for name, (keys, parent) in foreign_keys.items()
])

In [ ]:
coverage = {
    "Étudiants sans inscription": len(set(students.id_etudiant) - set(inscriptions.id_etudiant)),
    "Programmes sans inscription": len(set(programs.id_programme) - set(inscriptions.id_programme)),
    "Inscriptions sans évaluation": len(set(inscriptions.id_inscription) - set(evaluations.id_inscription)),
    "Inscriptions sans activité": len(set(inscriptions.id_inscription) - set(activities.id_inscription)),
}
pd.Series(coverage)

### Valeurs manquantes

In [ ]:
missing = pd.DataFrame([
    {"table": name, "column": column, "missing": int(table[column].isna().sum())}
    for name, table in tables.items()
    for column in table.columns
])
missing

In [ ]:
print("Inscriptions sans date de début :", int(inscriptions["date_debut"].isna().sum()))

In [ ]:
pd.crosstab(inscriptions['resultat_final'],inscriptions['date_annulation'].notna())

In [ ]:
pd.crosstab(evaluations['note'].isna(),evaluations['type_evaluation'])

In [ ]:
missing_eval = evaluations[evaluations['note'].isna()]
missing_eval = missing_eval.merge(
    inscriptions[
        ["id_inscription","resultat_final","date_debut","date_annulation"]
    ],
    on='id_inscription',
    how='left'
)
print(missing_eval["resultat_final"].value_counts(dropna=False))

### Validité des valeurs et cohérence temporelle

In [ ]:
for name, table in tables.items():
    numeric = table.select_dtypes(include="number")
    numeric = numeric.drop(columns=[c for c in numeric if c.startswith("id_")])
    if not numeric.empty:
        print(name)
        print(numeric.describe())

In [ ]:
cat = ['genre','type_evaluation','semestre','resultat_final','type_activite','niveau_scolaire','annee_scolaire']
for name,table in tables.items():
    print(f'La table {name}:')
    for col in cat:
        if col in table.columns:
            print(f'la variable {col} contient les valeurs : {table[col].unique()} \n')

In [ ]:
inscriptions_dates = merge_tables(inscriptions, students, programs)

err_annul_debut = inscriptions_dates[
    inscriptions_dates['date_annulation'].notna()
    & (inscriptions_dates['date_annulation'] < inscriptions_dates['date_debut'])
]
err_annul_prog = inscriptions_dates[
    inscriptions_dates['date_annulation'].notna()
    & (inscriptions_dates['date_annulation'] < inscriptions_dates['date_debut_programme'])
]
debut_avant_prog = inscriptions_dates[
    inscriptions_dates['date_debut'].notna()
    & (inscriptions_dates['date_debut'] < inscriptions_dates['date_debut_programme'])
]

print(f"Inscriptions avec date_annulation < date_debut : {len(err_annul_debut)}")
print(f"Inscriptions avec date_annulation < date_debut_programme (T_0) : {len(err_annul_prog)}")
print(f"Inscriptions avec date_debut < date_debut_programme (T_0) : {len(debut_avant_prog)}")

In [ ]:
eval_dates = evaluations.merge(
    inscriptions_dates[['id_inscription', 'date_debut', 'date_debut_programme', 'date_annulation']],
    on='id_inscription',
    how='left',
)

print(f"Evaluations avec date_soumission < date_debut : {(eval_dates['date_soumission'] < eval_dates['date_debut']).sum()}")
print(f"Evaluations avec date_echeance < date_debut : {(eval_dates['date_echeance'] < eval_dates['date_debut']).sum()}")
print(f"Evaluations avec date_soumission < date_debut_programme (T_0) : {(eval_dates['date_soumission'] < eval_dates['date_debut_programme']).sum()}")
print(f"Evaluations avec date_echeance < date_debut_programme (T_0) : {(eval_dates['date_echeance'] < eval_dates['date_debut_programme']).sum()}")
print(f"Evaluations avec date_soumission > date_annulation : {(eval_dates['date_soumission'] > eval_dates['date_annulation']).sum()}")

In [ ]:
act_dates = activities.merge(
    inscriptions_dates[['id_inscription', 'date_debut', 'date_debut_programme', 'date_annulation']],
    on='id_inscription',
    how='left',
)

print(f"Activites avec horodatage < date_debut : {(act_dates['horodatage'] < act_dates['date_debut']).sum()}")
print(f"Activites avec horodatage < date_debut_programme (T_0) : {(act_dates['horodatage'] < act_dates['date_debut_programme']).sum()}")
print(f"Activites avec horodatage > date_annulation : {(act_dates['horodatage'] > act_dates['date_annulation']).sum()}")

### Contrôles d’intégrité après nettoyage

On vérifie la cohérence des tables nettoyées avec les données brutes et les règles d’exclusion.

In [ ]:
for table, key in ids.values():
    assert table[key].notna().all() and table[key].is_unique, "Clé primaire invalide."
assert inscriptions.id_etudiant.isin(students.id_etudiant).all(), "Lien étudiant invalide."
assert inscriptions.id_programme.isin(programs.id_programme).all(), "Lien programme invalide."
assert evaluations.id_inscription.isin(inscriptions.id_inscription).all(), "Lien évaluation invalide."
assert activities.id_inscription.isin(inscriptions.id_inscription).all(), "Lien activité invalide."

clean_ins, clean_eval, clean_act = exclude(inscriptions_dates, evaluations, activities)
clean_ins = target(clean_ins)
stored = {
    "inscriptions": pd.read_parquet(CLEAN_INSCRIPTIONS),
    "evaluations": pd.read_parquet(CLEAN_EVALUATIONS),
    "activities": pd.read_parquet(CLEAN_ACTIVITES),
}
rebuilt = {"inscriptions": clean_ins, "evaluations": clean_eval, "activities": clean_act}
matching_tables = {
    name: table.reset_index(drop=True).equals(stored[name].reset_index(drop=True))
    for name, table in rebuilt.items()
}
assert all(matching_tables.values()), "Les tables nettoyées ne correspondent pas aux sources."
assert clean_eval.id_inscription.isin(clean_ins.id_inscription).all()
assert clean_act.id_inscription.isin(clean_ins.id_inscription).all()

data_summary = {
    "raw_enrollments": len(inscriptions),
    "cleaned_enrollments": len(clean_ins),
    "excluded_enrollments": len(inscriptions) - len(clean_ins),
    "raw_table_counts": {name: len(table) for name, table in raw.items()},
    "clean_table_counts": {name: len(table) for name, table in rebuilt.items()},
    "matching_tables": matching_tables,
}
DATA_SUMMARY.parent.mkdir(parents=True, exist_ok=True)
DATA_SUMMARY.write_text(json.dumps(data_summary, indent=2, allow_nan=False), encoding="utf-8")
pd.Series(data_summary["clean_table_counts"], name="Observations nettoyées")

## Conclusion

- Les clés primaires et étrangères des cinq tables sont valides. Les 7 169 inscriptions comportent au moins une trace d’activité sur l’ensemble de la période, tandis que 887 n’ont aucune évaluation enregistrée (12,4 %).
- Le champ `filiere` de la table `programmes` est entièrement vide et est retiré lors de la fusion. Les 686 échéances et les 48 notes manquantes sont conservées. Une inscription ne possède pas de date individuelle de début.
- Le nettoyage exclut 127 inscriptions annulées avant le début théorique du programme, 4 inscriptions portant le statut `Echec` malgré une annulation datée et 14 abandons sans date d’annulation confirmée.
- Les évaluations dont la soumission ou l’échéance précède la date individuelle de début sont également exclues.

Après nettoyage, **7 024 inscriptions**, **2 589 638 activités** et **42 264 évaluations** sont conservées. Sur les 48 évaluations retirées, 30 le sont pour incohérence chronologique. L’absence d’inscriptions n’ayant jamais eu d’activité limite la représentativité du jeu de données.